In [ ]:
import json
import random
import torch
import numpy as np
from tqdm import tqdm
from collections import defaultdict
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForCausalLM
import time
import re


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

In [ ]:
# with open("/content/drive/MyDrive/JokerData/joker_task1_retrieval_corpus25_EN.json") as f:
#     corpus = json.load(f)

# with open("/content/drive/MyDrive/JokerData/joker_task1_retrieval_queries_train25_EN.json") as f:
#     queries_train = json.load(f)

# with open("/content/drive/MyDrive/JokerData/joker_task1_retrieval_queries_test25_EN.json") as f:
#     queries_test = json.load(f)
    
# with open("/content/drive/MyDrive/JokerData/joker_task1_retrieval_qrels_train25_EN.json") as f:
#     qrels_train = json.load(f)

nr_obs = 10000


with open("/kaggle/input/jokerds/joker_task1_retrieval_corpus25_EN.json") as f:
    corpus = json.load(f)

with open("/kaggle/input/jokerds/joker_task1_retrieval_queries_train25_EN.json") as f:
    queries_train = json.load(f)

with open("/kaggle/input/jokerds/joker_task1_retrieval_queries_test25_EN.json") as f:
    queries_test = json.load(f)
    
with open("/kaggle/input/jokerds/joker_task1_retrieval_qrels_train25_EN.json") as f:
    qrels_train = json.load(f)


# create new datasets of size nr_obs, to be able to compute in time:
corpus = list(corpus[:nr_obs])
qrels_train = [d for d in qrels_train if d.get("docid") is not None and d["docid"] <= nr_obs]

print(corpus[0], queries_train[0], qrels_train[0])
print(len(corpus))


In [ ]:
# constructing the model the same way as in hugging face tutorial
model_id = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)


In [ ]:
# creating the messages afterwards:

few_shots_msg = [ {
          "role": "system",
          "content": (
              "You are a text classification system.\n"
              "Determine whether the text contains humor and whether it is related to the topic.\n"
              "Return JSON only."
          )
      },

      # Few-shot example 1
      {
          "role": "user",
          "content": (
              "Topic: death\n"
              "Text: \"The Easter story is not a dead issue.\""
          )
      },
      {
          "role": "assistant",
          "content": "{ \"contains_humor\": true, \"about_topic\": true }"
      },

      # Few-shot example 2
      {
          "role": "user",
          "content": (
              "Topic: space\n"
              "Text: \"Mars is the fourth planet from the Sun and has a thin atmosphere.\""
          )
      },
      {
          "role": "assistant",
          "content": "{ \"contains_humor\": false, \"about_topic\": true }"
      },

      # Few-shot example 3
      {
          "role": "user",
          "content": (
              "Topic: tom\n"
              "Text: \"''You resemble a goat,'' said Tom satirically.\""
          )
      },
      {
          "role": "assistant",
          "content": "{ \"contains_humor\": true, \"about_topic\": true }"
      },

      # Few-shot example 4
      {
          "role": "user",
          "content": (
              "Topic: sugar\n"
              "Text: \"I need to eat more, I am getting weak.\""
          )
      },
      {
          "role": "assistant",
          "content": "{ \"contains_humor\": false, \"about_topic\": false }"
      }]

def create_messages(usr_content):
  messages = [
      {
          "role": "system",
          "content": (
              "You are a text classification system.\n"
              "Determine whether the text contains humor and whether it is related to the topic.\n"
              "Return JSON only."
          )
      },

      # Few-shot example 1
      {
          "role": "user",
          "content": (
              "Topic: death\n"
              "Text: \"The Easter story is not a dead issue.\""
          )
      },
      {
          "role": "assistant",
          "content": "{ \"contains_humor\": true, \"about_topic\": true }"
      },

      # Few-shot example 2
      {
          "role": "user",
          "content": (
              "Topic: space\n"
              "Text: \"Mars is the fourth planet from the Sun and has a thin atmosphere.\""
          )
      },
      {
          "role": "assistant",
          "content": "{ \"contains_humor\": false, \"about_topic\": true }"
      },

      # Few-shot example 3
      {
          "role": "user",
          "content": (
              "Topic: tom\n"
              "Text: \"''You resemble a goat,'' said Tom satirically.\""
          )
      },
      {
          "role": "assistant",
          "content": "{ \"contains_humor\": true, \"about_topic\": true }"
      },

      # Few-shot example 4
      {
          "role": "user",
          "content": (
              "Topic: sugar\n"
              "Text: \"I need to eat more, I am getting weak.\""
          )
      },
      {
          "role": "assistant",
          "content": "{ \"contains_humor\": false, \"about_topic\": false }"
      },

      # Actual input
      {
          "role": "user",
          "content": user_content
      }
  ]

  return messages


def extract_first_json(text):
    """
    Extracts the first JSON object from a string.
    """
    match = re.search(r'\{.*?\}', text, re.DOTALL)
    if not match:
        raise ValueError("No JSON object found in model output")

    json_str = match.group(0)
    return json.loads(json_str)




def create_simple_message(topic, doc):
    message = f"Does the following text: {doc} contain wordplay and is relevant to the topic: {topic}? Respond with YES or NO."
    return message



counter = 1
times = []
use_simple_prompt = True

topic = {"qid":"19","query":"death"}
retrieved_ids = set()

topic_id = int(topic["qid"])
relevant_ids = set()

for rel in qrels_train:
    if int(rel["qid"]) == topic_id:
        # print("one add")
        relevant_ids.add(rel["docid"])

print(relevant_ids)

for text in corpus:
    # the topic and text that will be read from arrays 
    topic_str = topic["query"]
    text_str = (
        text["text"]
    )
    user_content = (
        f"Topic: {topic_str}\n"
        f"Text: \"{text_str}\""
    )

    if counter % 50 == 0:
        print(f"At counter {counter}, topic: {topic_str}, text: {text_str}")
    start = time.perf_counter()

    # use a simple prompt with only 1 sentence or a complex one using few shots example.
    if use_simple_prompt:
        prompt = create_simple_message(topic_str, text_str)

        # Tokenize
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
        # Generate
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,     # enough for YES or NO
            temperature=0.0,       # deterministic
            pad_token_id=tokenizer.eos_token_id
        )
        
        # Only decode the generated tokens (not the prompt)
        generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
        response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

        if counter % 50 == 0:
            print("Model response:", response)

        response = response.upper()
        if "YES" in response:
            retrieved_ids.add(int(text["docid"]))
       

        counter += 1
    else:

        system_tokens = tokenizer.apply_chat_template(
            few_shots_msg,
            tokenize=True,
            add_generation_prompt=True
        )
        
        # Apply safe conversion
        if isinstance(system_tokens, dict):
            system_input_ids = system_tokens["input_ids"]
        elif isinstance(system_tokens, list):
            # Convert list to tensor
            system_input_ids = torch.tensor(system_tokens, dtype=torch.long)
        else:
            system_input_ids = system_tokens  # already a tensor
        
        # Ensure 2D shape (batch_size, seq_len)
        if system_input_ids.dim() == 1:
            system_input_ids = system_input_ids.unsqueeze(0)
        
        system_input_ids = system_input_ids.to(model.device)
        
        # 2. Tokenize dynamic user message
        usr_msg = [{"role": "user", "content": user_content}]
        user_tokens = tokenizer.apply_chat_template(
            usr_msg,
            tokenize=True,
            add_generation_prompt=True
        )
        
        if isinstance(user_tokens, dict):
            user_input_ids = user_tokens["input_ids"]
        elif isinstance(user_tokens, list):
            user_input_ids = torch.tensor(user_tokens, dtype=torch.long)
        else:
            user_input_ids = user_tokens
        
        if user_input_ids.dim() == 1:
            user_input_ids = user_input_ids.unsqueeze(0)
        
        user_input_ids = user_input_ids.to(model.device)

        # user_input_ids = user_tokens["input_ids"].to(model.device)
        
        # 3. Concatenate
        input_ids = torch.cat([system_input_ids, user_input_ids], dim=-1)
        
        # 4. Generate
        outputs = model.generate(
            input_ids=input_ids,
            max_new_tokens=80,
            pad_token_id=tokenizer.eos_token_id
        )
        
        generated_ids = outputs[0][input_ids.shape[-1]:]
        raw_response = tokenizer.decode(generated_ids, skip_special_tokens=True)
        if counter % 50 == 0:
            print(raw_response)

        try:
            data = extract_first_json(raw_response)
        except ValueError as e:
            print("Failed to parse JSON:", raw_response)
            data = None
            counter += 1
            continue

        # data = json.loads(response)

        contains_humor = data.get("contains_humor", False)
        about_topic = data.get("about_topic", False)

        if contains_humor and about_topic:
            print(f"Topic and humor are both true, adding doc..., at counter {counter}")
            retrieved_ids.add(int(text["docid"]))

        counter += 1
        # --------------------------------------------------
        
        # messages = create_messages(user_content)
    
        # prompt = tokenizer.apply_chat_template(
        #   messages,
        #   tokenize=False,
        #   add_generation_prompt=True
        # )
    
        # inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
        # outputs = model.generate(
        #     **inputs,
        #     max_new_tokens=80,
        #     temperature=0.2,
        #     do_sample=True
        # )
    
        # Length of the prompt (in tokens)
        # prompt_len = inputs["input_ids"].shape[-1]
        
        # # Slice only the generated tokens
        # generated_ids = outputs[0][prompt_len:]
        
        # # Decode only the generated part
        # response = tokenizer.decode(generated_ids, skip_special_tokens=True)
    
        # # response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        # print(response)
        # data = json.loads(response)
        # contains_humor = data["contains_humor"]
        # about_topic = data["about_topic"]

        # if contains_humor and about_topic:
        #     print("Topic and humor are both true, adding doc...")
        #     retrieved_ids.add(int(text["docid"]))

    torch.cuda.synchronize()
    end = time.perf_counter()
    times.append(end - start)


print(retrieved_ids)

# Calculating precision, recall and F1-score for current topic
TP = len(retrieved_ids & relevant_ids)
FP = len(retrieved_ids - relevant_ids)
FN = len(relevant_ids - retrieved_ids)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
print("Precision:", precision)
print("Recall:", recall)
print("TP:", TP)
print("FN:", FN)
print("FP:", FP)
f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0 else 0.0
)
print("F1:", f1)
    
    

average_time = sum(times) / len(times)
print(f"Average inference time: {average_time:.3f} seconds")

In [ ]:
# import torch, gc
# gc.collect()
# torch.cuda.empty_cache()



# import torch
# import time

# start = time.perf_counter()

# # GPU work
# outputs = model.generate(...)

# torch.cuda.synchronize()  # IMPORTANT

# end = time.perf_counter()
# times.append(end - start)
# average_time = sum(times) / len(times)
# print(f"Average time: {average_time:.4f} seconds")